In [1]:
import os
import rasterio
import numpy as np
import pickle
import pandas as pd

# Define paths
sentinel2_path = 'D:/Kansas/data/10_counties_10km_matched_shapes/raw_data/satellite/'
gt_path = 'D:/Kansas/data/10_counties_10km_matched_shapes/raw_data/gt/'
output_path = 'D:/Kansas/data/10_counties_10km_matched_shapes/processed_data/All_bands_minmax_clipping_10k_equal_splits_8_months'
os.makedirs(output_path, exist_ok=True)

# Prepare lists for all counties
county_list = sorted(os.listdir(sentinel2_path))

# Function to apply Min-Max normalization
selected_bands = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]  # R, G, B, NIR
def minmax_normalize(sentinel_data):
    min_val, max_val = 0, 10000  # Sentinel-2 data range for reflectance
    sentinel_data = np.clip(sentinel_data, min_val, max_val)
    normalized_data = (sentinel_data - min_val) / (max_val - min_val)
    return normalized_data

def prepare_data(county):
    inputs, labels = [], []

    # Load the ground truth
    gt_image_path = os.path.join(gt_path, f'{county}.png')
    with rasterio.open(gt_image_path) as gt_src:
        gt_data = gt_src.read(1)

    # Load the sentinel images for the county
    sentinel2_county_path = os.path.join(sentinel2_path, county)
    sentinel_images = sorted(os.listdir(sentinel2_county_path))[:8]
    sentinel_data_list = []

    for img in sentinel_images:
        img_path = os.path.join(sentinel2_county_path, img)
        with rasterio.open(img_path) as src:
            sentinel_data = src.read(selected_bands)  # Select only RGB and NIR bands
            normalized_data = minmax_normalize(sentinel_data)  # Apply Min-Max normalization
            sentinel_data_list.append(normalized_data)

    # Convert lists to numpy arrays
    sentinel_data = np.stack(sentinel_data_list)  # Shape: (N, 4, H, W)

    # Ensure dimensions are correct for sampling
    H, W = gt_data.shape

    # Split data by thirds
    for y in range(H):
        for x in range(W):
            x_start, y_start = 3 * x, 3 * y
            sentinel_patch = sentinel_data[:, :, y_start:y_start+3, x_start:x_start+3]
            sentinel_flat = sentinel_patch.flatten()

            # Determine split assignment
            split = (y * W + x) / (H * W)  # Linear index normalized to [0, 1]
            if split < 1/3:
                split_name = 'train'
            elif split < 2/3:
                split_name = 'val'
            else:
                split_name = 'test'

            # Append the features and label
            inputs.append((sentinel_flat, split_name))
            labels.append((gt_data[y, x], split_name))

    # Save as numpy arrays for each county
    county_output_path = os.path.join(output_path, county)
    os.makedirs(county_output_path, exist_ok=True)

    for split in ['train', 'val', 'test']:
        split_inputs = np.array([inp for inp, s in inputs if s == split])
        split_labels = np.array([lab for lab, s in labels if s == split])
        np.save(os.path.join(county_output_path, f'{split}_inputs.npy'), split_inputs)
        np.save(os.path.join(county_output_path, f'{split}_labels.npy'), split_labels)

# Process data for each county
for county in county_list:
    prepare_data(county)

print("Data preparation completed successfully!")

C:\Users\udays\miniconda3\Lib\site-packages\rasterio\__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


Data preparation completed successfully!


In [1]:
import os
import numpy as np

# Define the paths
output_path = 'D:/Kansas/data/10_counties_10km_matched_shapes/processed_data/All_bands_minmax_clipping_10k_equal_splits_8_months'
combined_output_path = os.path.join(output_path, 'combined')
os.makedirs(combined_output_path, exist_ok=True)

# Initialize dictionaries to hold combined data for each split
combined_data = {
    'train': {'inputs': [], 'labels': []},
    'val': {'inputs': [], 'labels': []},
    'test': {'inputs': [], 'labels': []}
}

# Loop through each county directory
for county in sorted(os.listdir(output_path)):
    county_path = os.path.join(output_path, county)
    if not os.path.isdir(county_path):  # Skip if not a directory
        continue

    # Loop through splits (train, val, test)
    for split in ['train', 'val', 'test']:
        inputs_file = os.path.join(county_path, f'{split}_inputs.npy')
        labels_file = os.path.join(county_path, f'{split}_labels.npy')

        # Load and append data to the combined dataset
        if os.path.exists(inputs_file) and os.path.exists(labels_file):
            split_inputs = np.load(inputs_file)
            split_labels = np.load(labels_file)
            combined_data[split]['inputs'].append(split_inputs)
            combined_data[split]['labels'].append(split_labels)

# Combine and save the data for each split
for split in ['train', 'val', 'test']:
    # Concatenate all county data for this split
    combined_inputs = np.concatenate(combined_data[split]['inputs'], axis=0)
    combined_labels = np.concatenate(combined_data[split]['labels'], axis=0)

    # Save to combined output path
    np.save(os.path.join(combined_output_path, f'{split}_inputs.npy'), combined_inputs)
    np.save(os.path.join(combined_output_path, f'{split}_labels.npy'), combined_labels)

print("All county data combined successfully!")

All county data combined successfully!
